In [ ]:
#always run this first!!
#essential reticulate functions that allow us to use python packages in R
#you'll need a conda env with 'leidenalg' and 'pandas' installed to do this
#then route reticulate to the python installed in that conda env with the below functions
Sys.setenv(RETICULATE_PYTHON="/user/.conda/envs/reticulate/bin/python")
reticulate::use_python("/user/.conda/envs/reticulate/bin/python")
reticulate::use_condaenv("/user/.conda/envs/reticulate")
reticulate::py_module_available(module='leidenalg') #needs to be TRUE
reticulate::import('leidenalg') #good to make sure this doesn't error

suppressMessages(library(GenomicRanges))
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(tidyverse))
suppressMessages(library(knitr))
suppressMessages(library(SoupX))
suppressMessages(library(plyr))

suppressMessages(require(DoubletFinder))
opts_chunk$set(tidy=TRUE)
warnLevel <- getOption('warn')
options(warn = -1)

# Singlome data

### Read in raw data from cellranger, filter by min 500 genes, run doubletFinder (4% doublet rate), perform soupX using acinar marker genes, store seurat object for each sample

In [ ]:
samples <- c('MM_354','MM_355','MM_356','MM_357','MM_358','MM_359','MM_360','MM_380',
           'MM_381','MM_382','MM_398','MM_399','MM_401','MM_402','MM_403','MM_404',
           'MM_405','MM_406','MM_408','MM_361','MM_407','MM_400','MM_426','MM_397',
           'MM_553','MM_554','MM_555','MM_556','MM_557','MM_558','MM_559','MM_560','MM_561')

In [ ]:
options(repr.plot.width=12, repr.plot.height=6)
contam_frac_results <- NULL
#This block uses automated contamination estimates
for (sample in samples){
    print(c(sample, Sys.time()))

    wd <- sprintf('/cellranger_outputs/%s/outs', sample)
    name <- sample
    rna_counts <- Read10X_h5(file.path(wd, 'raw_feature_bc_matrix.h5'))
    data <- CreateSeuratObject(counts=rna_counts)
    print ("OG object")
    print(data)
    
    data[['percent.mt']] <- PercentageFeatureSet(data, pattern = '^MT-')
    data <- subset(x = data, subset = nFeature_RNA > 500)
    print ("Object filt for min Genes")
    print(data)
    
    data <- NormalizeData(data, normalization.method = "LogNormalize", scale.factor = 10000) #Ruth added
    data <- FindVariableFeatures(data, selection.method = "vst", nfeatures = 2000)
    all.genes <- rownames(data)
    data <- ScaleData(data, features = all.genes)
    
    data <- RunPCA(data, verbose = FALSE)
    data <- RunUMAP(data, dims = 1:30, verbose = FALSE)
    data <- FindNeighbors(data, dims = 1:30, verbose = FALSE)
    data <- FindClusters(data, algorithm=4, resolution = 1, verbose=FALSE)
    
    #options(repr.plot.height = 4, repr.plot.width = 6)
    p1 <- DimPlot(data, reduction = "umap", label = TRUE, pt.size = .1)
    print(p1)
    marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
                  'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
                  'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
                  'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
                  'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
                  'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
                  'NCR1','NCAM1','CD4','FOXP3')
    #options(repr.plot.width=12, repr.plot.height=6)
    p2 <- DotPlot(data, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
    p2 <- p2 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('pre soup expressiong') + ylab('')  
    print(p2)
    
    
    nExp <- round(ncol(data) * 0.04)  # expect 4% doublets

    data.filt <- doubletFinder_v3(data, pN = 0.25, pK = 0.09, nExp = nExp, PCs = 1:10)
    data.filt@meta.data
    doubFind_df <- data.filt@meta.data %>% select(tail(names(.), 2)) 
    colnames(doubFind_df) <- c('pANN','DoubletFinder')

    num_doublet = sum(doubFind_df$DoubletFinder == 'Doublet')
    print(paste("The number of doublets for", sample, " is ",num_doublet))
    #get the cutoff number (inefficient but not redoing anything and is at least record keeping)
    write.table(doubFind_df, file = sprintf('/data/%s_doubletFinder.txt', sample), row.names=TRUE, col.names=FALSE, quote=FALSE)
    #fin_scrub_df = rbind(fin_scrub_df,doubFind_df)
    
    
    doub <- read.table(sprintf('/data/%s_doubletFinder.txt', sample))

    doubFinder_doublets = doub[doub$V3=='Doublet',1]
    #head(scrub_doublets)
    ##create vector of assignments with BC names
    cells <- Cells(data)
    doubFinder_metadata = rep('Singlet', length(cells))
    names(doubFinder_metadata) <- cells
    doubFinder_metadata[doubFinder_doublets] <- 'Doublet'
    table(doubFinder_metadata)
    head(names(doubFinder_metadata))
    #
    ##add metadata and visualize on the combined UMAP
    data <- AddMetaData(data, doubFinder_metadata, col.name='DoubletFinder')
    #
    ##visualize amulet multiplets on the combined UMAP
    p3 <- DimPlot(data, reduction='umap', group.by = "DoubletFinder", pt.size = 1, raster=FALSE,
        cols = c("Singlet" = "grey", "Doublet" = "blue"), 
        order = c("Doublet", "Singlet"))
    print(p3)
    data <- subset(data, subset=DoubletFinder=="Singlet")
    #head(data@meta.data)
    print ("Object post doubletfinder")
    print(data)
    
     DefaultAssay(data) <- 'RNA'
    toc <- GetAssayData(object = data, slot = "counts") #with nFeature >500 filter
    tod <- Seurat::Read10X_h5(file.path(wd, 'raw_feature_bc_matrix.h5')) #raw count matrix
    
    metadata <- (cbind(as.data.frame(data[["umap"]]@cell.embeddings),
                   as.data.frame(Idents(data)),
                   as.data.frame(Idents(data))))
    colnames(metadata) <- c("RD1","RD2","Cluster","Annotation")
    
    sc <- SoupChannel(tod,toc)
    sc <- setDR(sc,metadata[colnames(sc$toc),c("RD1","RD2")])
    sc <- setClusters(sc,setNames(metadata$Cluster,rownames(metadata)))
    original <- sc$soupProfile[order(sc$soupProfile$est,decreasing=TRUE),]
    fwrite(x = original, file = sprintf("/data/%s_est_soupX.txt",sample), sep = '\t', row.names = TRUE, col.names = TRUE)

    useToEst <- estimateNonExpressingCells(sc, nonExpressedGeneList = list("REG1A","PRSS1") ,)
    sc <- calculateContaminationFraction(sc,list("REG1A","PRSS1"),useToEst=useToEst,forceAccept=TRUE)
    contamination_fraction <- mean(sc$metaData$rho*100)
    contam_frac_results$Sample <- name
    contam_frac_results$ContaminationFraction <- contamination_fraction
    out <- adjustCounts(sc)

    data2 <- CreateSeuratObject(out)
    data2[['percent.mt']] <- PercentageFeatureSet(data2, pattern = '^MT-')
    data2 <- NormalizeData(data2, normalization.method = "LogNormalize", scale.factor = 10000) #Ruth added
    data2 <- FindVariableFeatures(data2, selection.method = "vst", nfeatures = 2000)
    all.genes <- rownames(data2)
    data2 <- ScaleData(data2, features = all.genes)
    data2 <- RunPCA(data2, verbose = FALSE)
    data2 <- RunUMAP(data2, dims = 1:30, verbose = FALSE)
    data2 <- FindNeighbors(data2, dims = 1:30, verbose = FALSE)
    data2 <- FindClusters(data2, algorithm=4, resolution = 1, verbose=FALSE)
    
    p4 <- DimPlot(data2, reduction = "umap", label = TRUE, pt.size = .1)
    print(p4)
    marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
                  'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
                  'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
                  'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
                  'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
                  'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
                  'NCR1','NCAM1','CD4','FOXP3')
    #options(repr.plot.width=12, repr.plot.height=6)
    p5 <- DotPlot(data2, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
    p5 <- p5 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('post soup expression') + ylab('')# + title('Post Soup')
    print(p5)
    
    
    saveRDS(data2, file = sprintf('/data/%s_postqc_doubletFinder_soupXgeneSelect.rds', sample))
    }

## Multiome data

### Read in raw data from cellranger, filter by min 500 genes, filter by min 1000 atac fragments, remove multiplets, run doubletFinder (4% doublet rate), perform soupX using acinar marker genes, store seurat object for each sample

In [ ]:
samples <- c('MM_662','MM_664','MM_667','MM_666','MM_665','MM_660','MM_661','6229_sort')


In [ ]:
options(repr.plot.width=12, repr.plot.height=6)
contam_frac_results <- NULL
#This block uses automated contamination estimates
for (sample in samples){
    print(c(sample, Sys.time()))

    wd <- sprintf('/cellranger_outputs/%s/outs/', sample)
    name <- sample

    inputdata.10x <- Read10X_h5(file.path(wd, 'raw_feature_bc_matrix.h5'))

    rna_counts <- inputdata.10x$`Gene Expression`
    atac_counts <- inputdata.10x$Peaks
    data <- CreateSeuratObject(counts=rna_counts)
    data[['percent.mt']] <- PercentageFeatureSet(data, pattern = '^MT-')

    data
    
    grange.counts <- StringToGRanges(rownames(atac_counts), sep = c(':', '-'))
    grange.use <- seqnames(grange.counts) %in% standardChromosomes(grange.counts)
    atac_counts <- atac_counts[as.vector(grange.use), ]
    annotations <- GetGRangesFromEnsDb(ensdb=EnsDb.Hsapiens.v86)
    seqlevelsStyle(annotations) <- 'UCSC'
    genome(annotations) <- 'hg38'
    
    frag.file <- file.path(wd, 'atac_fragments.tsv.gz')
    suppressWarnings(chrom_assay <- CreateChromatinAssay(counts=atac_counts, sep=c(':', '-'), genome='hg38', min.cells = -1, min.features = -1, fragments=frag.file, annotation=annotations))
    data[['ATAC']] <- chrom_assay
    
    qc <- read.table(file.path(wd, 'per_barcode_metrics.csv'), sep=',', header=TRUE, stringsAsFactors=1)
    qc <- as.data.frame(qc)
    rownames(qc) <- qc$gex_barcode
    qc <- qc[Cells(data), 1:length(colnames(qc))]
    data <- AddMetaData(data, qc)
    print ("OG object")
    print(data)
    
    data <- subset(
      x = data,
      subset = nFeature_RNA > 500
    )
    print ("Object filt for min Genes")
    print(data)
    
    data <- subset(
      x = data,
      subset = atac_fragments > 1000 
    )
    print ("Object filt for min frag")
    print(data)
    
    data <- subset(
      x = data,
      subset = excluded_reason != 1 
    )
    print ("Object filt for multiplets")
    print(data)
    
    
    data <- NormalizeData(data, normalization.method = "LogNormalize", scale.factor = 10000) #Ruth added
    data <- FindVariableFeatures(data, selection.method = "vst", nfeatures = 2000)
    all.genes <- rownames(data)
    data <- ScaleData(data, features = all.genes)
    
    data <- RunPCA(data, verbose = FALSE)
    data <- RunUMAP(data, dims = 1:30, verbose = FALSE)
    data <- FindNeighbors(data, dims = 1:30, verbose = FALSE)
    data <- FindClusters(data, algorithm=4, resolution = 1, verbose=FALSE)
    
    #options(repr.plot.height = 4, repr.plot.width = 6)
    p1 <- DimPlot(data, reduction = "umap", label = TRUE, pt.size = .1)
    print(p1)
    marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
                  'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
                  'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
                  'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
                  'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
                  'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
                  'NCR1','NCAM1','CD4','FOXP3')
    #options(repr.plot.width=12, repr.plot.height=6)
    p2 <- DotPlot(data, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
    p2 <- p2 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('pre soup expressiong') + ylab('')  
    print(p2)
    
    
    nExp <- round(ncol(data) * 0.04)  # expect 4% doublets

    data.filt <- doubletFinder_v3(data, pN = 0.25, pK = 0.09, nExp = nExp, PCs = 1:10)
    data.filt@meta.data
    doubFind_df <- data.filt@meta.data %>% select(tail(names(.), 2))
    colnames(doubFind_df) <- c('pANN','DoubletFinder')

    num_doublet = sum(doubFind_df$DoubletFinder == 'Doublet')
    print(paste("The number of doublets for", sample, " is ",num_doublet))
    #get the cutoff number (inefficient but not redoing anything and is at least record keeping)
    write.table(doubFind_df, file = sprintf('/data/%s_doubletFinder.txt', sample), row.names=TRUE, col.names=FALSE, quote=FALSE)
    #fin_scrub_df = rbind(fin_scrub_df,doubFind_df)
    
    
    doub <- read.table(sprintf('/data/%s_doubletFinder.txt', sample))

    doubFinder_doublets = doub[doub$V3=='Doublet',1]
    #head(scrub_doublets)
    ##create vector of assignments with BC names
    cells <- Cells(data)
    doubFinder_metadata = rep('Singlet', length(cells))
    names(doubFinder_metadata) <- cells
    doubFinder_metadata[doubFinder_doublets] <- 'Doublet'
    table(doubFinder_metadata)
    head(names(doubFinder_metadata))
    #
    ##add metadata and visualize on the combined UMAP
    data <- AddMetaData(data, doubFinder_metadata, col.name='DoubletFinder')
    #
    ##visualize amulet multiplets on the combined UMAP
    p3 <- DimPlot(data, reduction='umap', group.by = "DoubletFinder", pt.size = 1, raster=FALSE,
        cols = c("Singlet" = "grey", "Doublet" = "blue"), 
        order = c("Doublet", "Singlet"))
    print(p3)
    data <- subset(data, subset=DoubletFinder=="Singlet")
    #head(data@meta.data)
    print ("Object post doubletfinder")
    print(data)
    
    DefaultAssay(data) <- 'RNA'
    toc <- GetAssayData(object = data, slot = "counts") #with nFeature >500 filter
    tod <- Seurat::Read10X_h5(file.path(wd, 'raw_feature_bc_matrix.h5'))$`Gene Expression` #raw count matrix
    
    metadata <- (cbind(as.data.frame(data[["umap"]]@cell.embeddings),
                   as.data.frame(Idents(data)),
                   as.data.frame(Idents(data))))
    colnames(metadata) <- c("RD1","RD2","Cluster","Annotation")
    
    sc <- SoupChannel(tod,toc)
    sc <- setDR(sc,metadata[colnames(sc$toc),c("RD1","RD2")])
    sc <- setClusters(sc,setNames(metadata$Cluster,rownames(metadata)))
    original <- sc$soupProfile[order(sc$soupProfile$est,decreasing=TRUE),]
    fwrite(x = original, file = sprintf("/data/%s_est_soupX.txt",sample), sep = '\t', row.names = TRUE, col.names = TRUE)

    useToEst <- estimateNonExpressingCells(sc, nonExpressedGeneList = list("REG1A","PRSS1") ,)
    sc <- calculateContaminationFraction(sc,list("REG1A","PRSS1"),useToEst=useToEst,forceAccept=TRUE)
    contamination_fraction <- mean(sc$metaData$rho*100)
    contam_frac_results$Sample <- name
    contam_frac_results$ContaminationFraction <- contamination_fraction
    out <- adjustCounts(sc)

    data2 <- CreateSeuratObject(out)
    data2[['percent.mt']] <- PercentageFeatureSet(data2, pattern = '^MT-')
    data2 <- NormalizeData(data2, normalization.method = "LogNormalize", scale.factor = 10000) #Ruth added
    data2 <- FindVariableFeatures(data2, selection.method = "vst", nfeatures = 2000)
    all.genes <- rownames(data2)
    data2 <- ScaleData(data2, features = all.genes)
    data2 <- RunPCA(data2, verbose = FALSE)
    data2 <- RunUMAP(data2, dims = 1:30, verbose = FALSE)
    data2 <- FindNeighbors(data2, dims = 1:30, verbose = FALSE)
    data2 <- FindClusters(data2, algorithm=4, resolution = 1, verbose=FALSE)
    
    p4 <- DimPlot(data2, reduction = "umap", label = TRUE, pt.size = .1)
    print(p4)
    marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
                  'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
                  'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
                  'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
                  'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
                  'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
                  'NCR1','NCAM1','CD4','FOXP3')
    #options(repr.plot.width=12, repr.plot.height=6)
    p5 <- DotPlot(data2, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
    p5 <- p5 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('post soup expression') + ylab('')# + title('Post Soup')
    print(p5)
    
    
    saveRDS(data2, file = sprintf('/data/%s_postqc_doubletFinder_soupXgeneSelect.rds', sample))
    }